# FASE 2 — lanzador reproducible de experimentos

Este notebook clona la rama de trabajo, instala el entorno administrado y ejecuta **un experimento preregistrado** o la suite de verificación. Está diseñado para usar **Entorno de ejecución → Ejecutar todo**.

No solicita tokens ni credenciales. Los resultados se empaquetan en un ZIP descargable. Para experimentos contextuales futuros, selecciona primero un entorno GPU en Colab.


In [ ]:
#@title 1. Configuración (normalmente solo hay que pulsar «Ejecutar todo»)
EXPERIMENTO = "suite_pruebas" #@param ["suite_pruebas", "diagnostico_gpu", "palabras_caracteres", "extension_representaciones", "ensamble_calibrado"]
DESCARGAR_ZIP = True #@param {type:"boolean"}
RAMA = "arena/01a0b014-fase-2"
REPO = "https://github.com/joako0o/FASE_2.git"
print({"experimento": EXPERIMENTO, "rama": RAMA, "descargar": DESCARGAR_ZIP})


In [ ]:
#@title 2. Clonar/actualizar e instalar el entorno administrado
import os, shutil, subprocess, sys
from pathlib import Path
BASE = Path("/content").resolve()
PROYECTO = BASE / "FASE_2"
BASE.mkdir(parents=True, exist_ok=True)
# Es imprescindible salir del checkout antes de borrarlo al reejecutar el notebook.
os.chdir(BASE)
if PROYECTO.exists():
    if PROYECTO.parent != BASE:
        raise RuntimeError(f"Ruta de proyecto insegura: {PROYECTO}")
    shutil.rmtree(PROYECTO)
subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", RAMA,
                REPO, str(PROYECTO)], cwd=BASE, check=True)
os.chdir(PROYECTO)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROYECTO, check=True,
                        text=True, stdout=subprocess.PIPE).stdout.strip()
subprocess.run([sys.executable, "scripts/40_gestionar_proyecto.py", "instalar"],
               cwd=PROYECTO, check=True)
PYTHON = str(PROYECTO / ".venv/bin/python")
RESULTADOS = BASE / "resultados_fase2" / EXPERIMENTO
RESULTADOS.mkdir(parents=True, exist_ok=True)
print("Proyecto:", PROYECTO)
print("Rama:", RAMA)
print("Commit:", commit)
print("Python administrado:", PYTHON)


In [ ]:
#@title 3. Diagnóstico de hardware y ejecución
import json, os, subprocess, sys
from pathlib import Path

env = os.environ.copy()
env.update({"OPENBLAS_NUM_THREADS":"1", "OMP_NUM_THREADS":"1", "TOKENIZERS_PARALLELISM":"false"})
log_path = RESULTADOS / "ejecucion.log"

def run_capture(command):
    print("Ejecutando:", " ".join(map(str, command)), flush=True)
    proc = subprocess.run(command, cwd=PROYECTO, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path.write_text(proc.stdout, encoding="utf-8")
    print(proc.stdout)
    if proc.returncode:
        raise subprocess.CalledProcessError(proc.returncode, command)

if EXPERIMENTO == "diagnostico_gpu":
    info = {"python": sys.version, "cpu": os.cpu_count()}
    try:
        import torch
        info.update({"torch": torch.__version__, "cuda_disponible": torch.cuda.is_available(),
                     "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
    except Exception as exc:
        info.update({"torch": None, "error_torch": repr(exc)})
    try:
        info["nvidia_smi"] = subprocess.run(["nvidia-smi"], text=True, stdout=subprocess.PIPE,
                                             stderr=subprocess.STDOUT).stdout
    except Exception as exc:
        info["nvidia_smi"] = repr(exc)
    (RESULTADOS / "diagnostico.json").write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(info, indent=2, ensure_ascii=False))
elif EXPERIMENTO == "suite_pruebas":
    tests = ["tests.test_ensamble_calibrado_v3", "tests.test_extension_representaciones_v3",
             "tests.test_encoder_factibilidad_v3", "tests.test_adaptacion_dominio_bcch_v3",
             "tests.test_jerarquia_direccional_v3", "tests.test_palabras_caracteres_v3",
             "tests.test_ampliacion_pre2000_v3", "tests.test_adjudicar_pre2000_v3",
             "tests.test_preparar_anotacion_300_v3"]
    run_capture([PYTHON, "-m", "unittest", *tests, "-v"])
else:
    modules = {"palabras_caracteres": ("evaluar_palabras_caracteres_v3", "run"),
               "extension_representaciones": ("evaluar_extension_representaciones_v3", "run"),
               "ensamble_calibrado": ("evaluar_ensamble_calibrado_v3", "run")}
    module, function = modules[EXPERIMENTO]
    launcher = f'import json,sys; from pathlib import Path; sys.path.insert(0,"scripts"); import {module} as m; print(json.dumps(m.{function}(Path(r"{RESULTADOS}")),ensure_ascii=False,indent=2))'
    run_capture([PYTHON, "-c", launcher])
print("Resultados en:", RESULTADOS)


In [ ]:
#@title 4. Verificar y descargar resultados
import hashlib, json, shutil
files = [p for p in RESULTADOS.rglob("*") if p.is_file()]
checksums = {str(p.relative_to(RESULTADOS)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
(RESULTADOS / "checksums_colab.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")
zip_path = shutil.make_archive(str(RESULTADOS), "zip", root_dir=RESULTADOS)
print("ZIP:", zip_path)
print("Archivos:", len(files) + 1)
if DESCARGAR_ZIP:
    from google.colab import files as colab_files
    colab_files.download(zip_path)


## Cómo devolver un experimento

Sube el ZIP descargado a la conversación. Contiene logs, manifiestos y checksums. No pegues credenciales ni habilites acceso de escritura a GitHub.

Los resultados de los experimentos existentes son reproducciones; cualquier experimento nuevo debe agregarse primero al repositorio con su protocolo preregistrado.
